In [41]:
# Step 1
import pandas as pd

file_path = "NAPLAN national results dataset.xlsx"

df = pd.read_excel(file_path, sheet_name="NAPLAN Results")

print(df.columns.tolist())
print(df.head())

['Calendar year', 'Year level', 'Domain', 'State/territory', 'Student attribute', 'Subgroup', 'Reporting flag', 'Enrolled students', 'Participation (%)', 'Participation (Num)', 'Average NAPLAN score', 'Average NAPLAN score confidence interval', 'NAPLAN score standard deviation', 'Exempt (%)', 'Needs additional support (%)', 'Developing (%)', 'Strong (%)', 'Exceeding (%)', 'Exceeding (CI)', 'Strong or above (%)', 'Strong or above (CI)', 'Developing or above (%)', 'Developing or above (CI)']
   Calendar year  Year level   Domain  State/territory Student attribute  \
0           2025           3  Reading  New South Wales               All   
1           2025           3  Reading  New South Wales            Gender   
2           2025           3  Reading  New South Wales            Gender   
3           2025           3  Reading  New South Wales       Indigeneity   
4           2025           3  Reading  New South Wales       Indigeneity   

         Subgroup Reporting flag  Enrolled stude

In [43]:
# Step 2
df = df.rename(columns={
    "Calendar year": "year",
    "Year level": "year_level",
    "Domain": "domain",
    "State/territory": "state",
    "Student attribute": "attribute",
    "Subgroup": "subgroup",
    "Reporting flag": "reporting_flag",
    "Participation (%)": "participation_pct",
    "Average NAPLAN score": "avg_score",
    "Needs additional support (%)": "needs_support_pct",
    "Developing (%)": "developing_pct",
    "Strong (%)": "strong_pct",
    "Exceeding (%)": "exceeding_pct",
    "Strong or above (%)": "strong_or_above_pct",
    "Developing or above (%)": "developing_or_above_pct"
})

In [45]:
# Step 3
print("Domains:")
print(df["domain"].dropna().unique())

print("\nAttributes:")
print(df["attribute"].dropna().unique())

Domains:
['Reading' 'Writing' 'Spelling' 'Grammar and Punctuation' 'Numeracy']

Attributes:
['All' 'Gender' 'Indigeneity' 'ABS remoteness' 'Language background'
 'Parental education' 'Parental occupation'
 'Indigeneity by ABS remoteness']


In [47]:
# Step 4
target_years = [2023, 2024, 2025]

target_domains = [
    "Reading",
    "Writing",
    "Numeracy",
    "Spelling",
    "Grammar and Punctuation"
]

df_filtered = df[
    df["year"].isin(target_years) &
    df["domain"].isin(target_domains)
].copy()

In [49]:
# Step 5
# 去掉报告受限或缺失的行
df_filtered = df_filtered[
    df_filtered["reporting_flag"].isna() |
    (df_filtered["reporting_flag"] == "")
].copy()

# 转数值
df_filtered["needs_support_pct"] = pd.to_numeric(df_filtered["needs_support_pct"], errors="coerce")

# 去掉主指标缺失
df_filtered = df_filtered.dropna(subset=["needs_support_pct"])

In [51]:
print(df["reporting_flag"].value_counts(dropna=False))

reporting_flag
NaN     16108
-        1260
n.p.      522
^         220
s.p.      180
#          70
Name: count, dtype: int64


In [53]:
target_attributes = [
    "All",
    "Gender",
    "Indigeneity",
    "ABS remoteness",
    "Language background",
    "Parental education",
    "Parental occupation"
]

df_filtered = df_filtered[df_filtered["attribute"].isin(target_attributes)].copy()

In [55]:
# Step 7
df_wide = df_filtered.pivot_table(
    index=["year", "year_level", "state", "attribute", "subgroup"],
    columns="domain",
    values="needs_support_pct",
    aggfunc="first"
).reset_index()

In [57]:
# Step 8
df_wide = df_wide.rename(columns={
    "Reading": "reading_nas",
    "Writing": "writing_nas",
    "Numeracy": "numeracy_nas",
    "Spelling": "spelling_nas",
    "Grammar and Punctuation": "grammar_punctuation_nas"
})

In [59]:
# Step 9
analysis_cols = [
    "reading_nas",
    "writing_nas",
    "spelling_nas",
    "grammar_punctuation_nas",
    "numeracy_nas"
]

df_analysis = df_wide.dropna(subset=analysis_cols).copy()

print(df_analysis.head())
print(df_analysis.shape)

domain  year  year_level      state       attribute        subgroup  \
0       2023           3  Australia  ABS remoteness  Inner regional   
1       2023           3  Australia  ABS remoteness    Major cities   
2       2023           3  Australia  ABS remoteness  Outer regional   
3       2023           3  Australia  ABS remoteness          Remote   
4       2023           3  Australia  ABS remoteness     Very remote   

domain  grammar_punctuation_nas  numeracy_nas  reading_nas  spelling_nas  \
0                     17.188876     11.677124    11.317540     14.917759   
1                     10.553130      8.169079     7.117291      8.679344   
2                     21.895964     16.147196    15.357964     18.214257   
3                     32.400297     26.688135    24.478573     28.109983   
4                     58.334510     54.220183    49.044092     53.316867   

domain  writing_nas  
0          7.528938  
1          4.607834  
2         10.651390  
3         20.262246  
4     

In [63]:
print(df_analysis.columns.tolist())
print(df_analysis.shape)
print(df_analysis.head(10))

['year', 'year_level', 'state', 'attribute', 'subgroup', 'grammar_punctuation_nas', 'numeracy_nas', 'reading_nas', 'spelling_nas', 'writing_nas']
(2414, 10)
domain  year  year_level      state       attribute        subgroup  \
0       2023           3  Australia  ABS remoteness  Inner regional   
1       2023           3  Australia  ABS remoteness    Major cities   
2       2023           3  Australia  ABS remoteness  Outer regional   
3       2023           3  Australia  ABS remoteness          Remote   
4       2023           3  Australia  ABS remoteness     Very remote   
5       2023           3  Australia             All             All   
6       2023           3  Australia          Gender          Female   
7       2023           3  Australia          Gender            Male   
8       2023           3  Australia     Indigeneity      Indigenous   
9       2023           3  Australia     Indigeneity  Non-Indigenous   

domain  grammar_punctuation_nas  numeracy_nas  reading_nas  s

In [65]:
df_analysis.columns.name = None
print(df_analysis.columns.tolist())
print(df_analysis.head())

['year', 'year_level', 'state', 'attribute', 'subgroup', 'grammar_punctuation_nas', 'numeracy_nas', 'reading_nas', 'spelling_nas', 'writing_nas']
   year  year_level      state       attribute        subgroup  \
0  2023           3  Australia  ABS remoteness  Inner regional   
1  2023           3  Australia  ABS remoteness    Major cities   
2  2023           3  Australia  ABS remoteness  Outer regional   
3  2023           3  Australia  ABS remoteness          Remote   
4  2023           3  Australia  ABS remoteness     Very remote   

   grammar_punctuation_nas  numeracy_nas  reading_nas  spelling_nas  \
0                17.188876     11.677124    11.317540     14.917759   
1                10.553130      8.169079     7.117291      8.679344   
2                21.895964     16.147196    15.357964     18.214257   
3                32.400297     26.688135    24.478573     28.109983   
4                58.334510     54.220183    49.044092     53.316867   

   writing_nas  
0     7.52893

In [67]:
# Step 10
df_analysis.to_csv("naplan_analysis_ready.csv", index=False)